In [ ]:
import joblib
import pandas as pd
import numpy as np

In [ ]:
als_model = joblib.load("../models/als_model.pkl")

user_item = joblib.load("../models/user_item_matrix.pkl")

user_to_idx = joblib.load("../models/user_to_idx.pkl")
song_to_idx = joblib.load("../models/song_to_idx.pkl")
idx_to_song = joblib.load("../models/idx_to_song.pkl")

content_model = joblib.load("../models/content_model.pkl")
scaler = joblib.load("../models/scaler.pkl")

song_features = pd.read_csv("../models/song_features.csv")

In [ ]:
feature_cols = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo"
]

X = song_features[feature_cols]
X_scaled = scaler.transform(X)

songid_to_content_idx = {
    sid: i for i, sid in enumerate(song_features["song_id"])
}

# Test MF Recommendation

In [ ]:
def recommend_cf(user_id, N=10):
    if user_id not in user_to_idx:
        raise ValueError("User not found")

    uidx = user_to_idx[user_id]

    rec_item_idxs, scores = als_model.recommend(
        uidx,
        user_item[uidx],
        N=N,
        filter_already_liked_items=True
    )

    rec_song_ids = [idx_to_song[int(i)] for i in rec_item_idxs]

    return pd.DataFrame({
        "song_id": rec_song_ids,
        "score": scores
    })

# Test Content-based Recommendation

In [ ]:
def recommend_content(song_id, N=10):
    if song_id not in songid_to_content_idx:
        raise ValueError("Song not found in content dataset")

    idx = songid_to_content_idx[song_id]

    distances, indices = content_model.kneighbors(
        X_scaled[idx].reshape(1, -1),
        n_neighbors=N + 1
    )

    rec_idxs = indices[0][1:]
    rec_distances = distances[0][1:]

    rec_df = song_features.iloc[rec_idxs][
        ["song_id", "title_x", "artist_name_x", "track_genre"]
    ].copy()

    rec_df["similarity"] = 1 - rec_distances
    return rec_df

# Evaluate Content Model

- Example recommendations
- GenreConsistency@K
- Average similarity score

In [ ]:
example_song_id = song_features.iloc[0]["song_id"]

query_song = song_features[song_features["song_id"] == example_song_id][
    ["song_id", "title_x", "artist_name_x", "track_genre"]
]
print("Query song:")
display(query_song)

print("Recommended similar songs:")
display(recommend_content(example_song_id, N=5))

### Genre consistency metric

In [ ]:
def genre_consistency(n_samples=100, top_k=5):
    sampled = song_features.sample(n=min(n_samples, len(song_features)), random_state=42)

    total = 0
    same_genre = 0

    for _, row in sampled.iterrows():
        recs = recommend_content(row["song_id"], N=top_k)
        same_genre += (recs["track_genre"] == row["track_genre"]).sum()
        total += len(recs)

    return same_genre / total
genre_score = genre_consistency(n_samples=100, top_k=5)
print("Genre consistency@5:", genre_score)

### Average similarity score

In [ ]:
def average_similarity_score(n_samples=100, top_k=5):
    sampled = song_features.sample(n=min(n_samples, len(song_features)), random_state=42)
    sims = []

    for _, row in sampled.iterrows():
        recs = recommend_content(row["song_id"], N=top_k)
        sims.extend(recs["similarity"].tolist())

    return np.mean(sims)
avg_sim = average_similarity_score(n_samples=100, top_k=5)
print("Average similarity@5:", avg_sim)

### Summary
The content-based recommender was evaluated using genre consistency and feature similarity metrics. Genre consistency measures how often the recommended songs share the same genre as the query song, while feature similarity measures the cosine similarity between the audio feature vectors of the query and recommended songs.

Across randomly sampled songs, the system achieved a GenreConsistency@5 of 0.372, indicating that approximately 37.2% of the recommended songs share the same genre as the query song. This suggests that the model captures meaningful musical characteristics that often align with genre categories.

In addition, the AverageSimilarity@5 was 0.967, showing that recommended songs are extremely close to the query song in the audio feature space. Since the recommender uses cosine similarity over normalized audio features (such as danceability, energy, acousticness, and tempo), this high similarity score confirms that the system successfully retrieves songs with highly similar musical attributes.

Overall, these results indicate that the content-based recommender effectively identifies songs with similar acoustic properties and frequently recommends songs within the same musical style.

# Evaluate MF model

In [ ]:
example_user = list(user_to_idx.keys())[100]

cf_recs = recommend_cf(example_user, N=100)

cf_recs_with_meta = cf_recs.merge(
    song_features[["song_id", "title_x", "artist_name_x", "track_genre"]],
    on="song_id",
    how="left"
)

display(
    cf_recs_with_meta[
        ["song_id", "title_x", "artist_name_x", "track_genre", "score"]
    ].head(20)
)

In [ ]:
cf_recs = recommend_cf(example_user, N=100)

cf_recs = cf_recs[
    cf_recs["song_id"].isin(song_features["song_id"])
]

cf_recs = cf_recs.head(10)

cf_recs_with_meta = cf_recs.merge(
    song_features[["song_id","title_x","artist_name_x","track_genre"]],
    on="song_id",
    how="left"
)
display(cf_recs_with_meta)

### Coverage in content catalog

In [ ]:
def cf_overlap_with_content(user_id, N=100):
    recs = recommend_cf(user_id, N=N)
    overlap = recs["song_id"].isin(songid_to_content_idx.keys()).sum()
    return overlap, N
overlap, total = cf_overlap_with_content(example_user, N=100)
print(f"Overlap with content catalog: {overlap}/{total}")

In [ ]:
import random
import numpy as np
from scipy.sparse import csr_matrix


eval_users = random.sample(list(user_to_idx.keys()), 1000)


def evaluate_hit_rate_at_k(model, user_item, user_to_idx, K=10, n_users=500, seed=42):
    rng = random.Random(seed)
    users = rng.sample(list(user_to_idx.keys()), min(n_users, len(user_to_idx)))

    hits = 0
    total = 0

    for user_id in users:
        uidx = user_to_idx[user_id]
        user_row = user_item[uidx]

        interacted_items = user_row.indices
        interacted_data = user_row.data

        if len(interacted_items) < 2:
            continue

        hidden_item = rng.choice(list(interacted_items))

        keep_mask = interacted_items != hidden_item
        kept_items = interacted_items[keep_mask]
        kept_data = interacted_data[keep_mask]

        if len(kept_items) == 0:
            continue

        user_row_copy = csr_matrix(
            (kept_data, ([0] * len(kept_items), kept_items)),
            shape=(1, user_item.shape[1])
        )

        rec_items, _ = model.recommend(
            userid=uidx,
            user_items=user_row_copy,
            N=K,
            filter_already_liked_items=True
        )

        if hidden_item in rec_items:
            hits += 1

        total += 1

    hit_rate = hits / total if total > 0 else 0.0
    print(f"HitRate@{K}: {hit_rate:.4f} ({hits}/{total})")
    return hit_rate

def evaluate_recall_at_k(model, user_item, user_to_idx, K=10, n_users=500, seed=42):
    rng = random.Random(seed)
    users = rng.sample(list(user_to_idx.keys()), min(n_users, len(user_to_idx)))

    recalls = []

    for user_id in users:
        uidx = user_to_idx[user_id]
        user_row = user_item[uidx]

        interacted_items = user_row.indices
        interacted_data = user_row.data

        if len(interacted_items) < 5:
            continue

        n_hide = max(1, int(len(interacted_items) * 0.2))
        hidden_items = set(rng.sample(list(interacted_items), n_hide))

        keep_mask = np.array([i not in hidden_items for i in interacted_items])
        kept_items = interacted_items[keep_mask]
        kept_data = interacted_data[keep_mask]

        if len(kept_items) == 0:
            continue

        user_row_copy = csr_matrix(
            (kept_data, ([0] * len(kept_items), kept_items)),
            shape=(1, user_item.shape[1])
        )

        rec_items, _ = model.recommend(
            userid=uidx,
            user_items=user_row_copy,
            N=K,
            filter_already_liked_items=True
        )

        hits = len(hidden_items.intersection(set(rec_items)))
        recalls.append(hits / len(hidden_items))

    recall_k = sum(recalls) / len(recalls) if recalls else 0.0
    print(f"Recall@{K}: {recall_k:.4f} ({len(recalls)} users)")
    return recall_k

Ks = [5, 10, 20, 50, 100]
hit_rates = []
recalls = []

for k in Ks:
    print(f"\nEvaluating K={k}")

    hr = evaluate_hit_rate_at_k(
        als_model,
        user_item,
        user_to_idx,
        K=k,
        n_users=1000
    )

    rc = evaluate_recall_at_k(
        als_model,
        user_item,
        user_to_idx,
        K=k,
        n_users=1000
    )

    hit_rates.append(hr)
    recalls.append(rc)


In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8,5))

plt.plot(Ks, hit_rates, marker='o', linewidth=2, label="HitRate@K")
plt.plot(Ks, recalls, marker='s', linewidth=2, label="Recall@K")

plt.xticks(Ks)

plt.xlabel("Top-K Recommendations")
plt.ylabel("Score")
plt.title("MF Recommendation Performance")

plt.legend()
plt.grid(alpha=0.3)

plt.show()

## Summary:
The collaborative filtering model was evaluated using HitRate@K and Recall@K under a leave-one-out protocol. For each user, one interaction was removed from their listening history and the model was asked to generate the top-K recommendations. A hit occurs when the hidden item appears within the top-K list.

The results are shown in Figure X, where both HitRate@K and Recall@K increase steadily as K grows. For small recommendation lists, the probability of recovering the hidden song is relatively low. For example, HitRate@5 is approximately 0.09, meaning that the hidden song appears in the top-5 recommendations for about 9% of users. As K increases, the performance improves substantially, reaching approximately 0.35 for HitRate@100.

This trend is expected because the Taste Profile dataset contains a very large item catalog (hundreds of thousands of songs) and highly sparse user interactions. Predicting the exact song a user previously listened to is therefore a difficult task. Despite this challenge, the model demonstrates meaningful ranking ability, as the hidden items increasingly appear in larger recommendation lists.

Overall, the results indicate that the matrix factorization model successfully captures latent relationships between users and songs, enabling it to recover previously interacted items within the recommended set.